# LABORATORIO 3

## Ejercicio 1

### Razonamiento previo


- Firma Digital vs Cifrado: El cifrado protege la confidencialidad (solo el destinatario con la clave privada puede leer el mensaje). La firma digital protege la autenticidad, la integridad y el no repudio (garantiza que el mensaje fue enviado por el dueño de la clave privada y que no ha sido alterado).

- El proceso de firmar: El emisor calcula una función hash del mensaje (una huella digital única) y cifra ese hash utilizando su clave privada.

- El proceso de verificar: El receptor descifra la firma utilizando la clave pública del emisor para recuperar el hash original, y lo compara con el hash calculado localmente sobre el mensaje recibido. Si ambos hashes coinciden exactamente, la firma es válida.-

In [2]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding

# 1. Generación de claves RSA (2048 bits es el estándar seguro actual)
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key()

# 2. Mensaje original
message = b"Transfer 100 euros"

# 3. FIRMA DIGITAL: Se usa la clave privada, padding PKCS1v15 y hash SHA256
signature = private_key.sign(
    message,
    padding.PKCS1v15(),
    hashes.SHA256()
)

print("¡Claves generadas y mensaje firmado con éxito!")
print(
    f"Mensaje original: {message.decode()}\n"
    f"Firma digital (en bytes): {signature}\n"
    f"Firma digital (en hexadecimal): {signature.hex()}"
)

¡Claves generadas y mensaje firmado con éxito!
Mensaje original: Transfer 100 euros
Firma digital (en bytes): b'\x05\xa5\xad\xc1\xb9\x17\x90\xc2H\x0b\x82X\x99\xdc\xdc\xdf\x00\x95\x966\xe2j\xee\xf8\x96\xbb\xe4\xce\x9e\xdb\x10\xca\xb5\x9ep\xb2P\xd3\xcf$\xa1Q\xa5\xdd2\x92\xb5\x9f_~\xc6\x96\xe8T\x15Qf\xc0\xd3%\xbf\xfbp\xa0\xe2 \'\xb6>\x17\x19\xd0\xe7\x9f\x1fX\x13\x1b\x0f\x1e%#;\x9f\xfe\x11\x91e\xdd?\x81I\x91i\xae\x92\xc6\x99\xa4 \xd0\x80"w\x0f\xd2K\xddy\xc8K$\x81$\xb5!\xb1\xdb\x84\x15\xca\xd2I\x88\xb9\x87\x16)l\xf7\xf1\x06\x9f\xdae=\xea\xde\xf7\xf7\x9b\xed\xad\xd2#\xaf\xa2\x96/\xe72\xda.\xb0G\x07\n\xeai\x19\xaa\\\x01\x9f\xbdq\xca\xcc\xe5t\x13\x13qZK\xf2t\x93\xb0\x90\xd7Q\xf6E*\xc4\xe0\xc6S\x9a\xa5\x06#A\xf5lV\xec\xba\xea>UW\xda\xf7\xaf\xc5\x96\xa0\x86\x85\xbd\x1c\xa6$\x90B\xd5\xff\x1f\t\xc4jc\xb1\xa9\x17N~\xd9\xb3X)\xccWl=\x90\xd6p\x03\xc9\x81 (\x82\xa1\xee\x85\xf2\xd3\x84l2_\xd9'
Firma digital (en hexadecimal): 05a5adc1b91790c2480b825899dcdcdf00959636e26aeef896bbe4ce9edb10cab59e70b250d3cf

a) ¿Qué hace este código en términos generales?

- Generación de claves: Crea un par de claves asimétricas RSA (una privada para firmar y otra pública para verificar).

- Firma: Toma el mensaje "Transfer 100 euros", le aplica una función hash (SHA256) y un esquema de relleno (PKCS1v15), y cifra el resultado con la clave privada para generar la signature.

- Verificación (con errores en el planteamiento original): Intenta comprobar la autenticidad de la firma utilizando la clave pública, pero introduce cambios en el proceso de validación (message + b" extra" y hashes.SHA1()).

b) ¿Qué problema hay en la verificación del mensaje? ¿Qué ataque podría ocurrir?

- El problema: Hay dos discrepancias críticas entre la firma y la verificación en el código original: Se firmó el mensaje message, pero en la verificación se evalúa message + b" extra" (el mensaje ha sido alterado/modificado en la entrada). Se firmó utilizando la función hash hashes.SHA256(), pero se intenta verificar utilizando hashes.SHA1().

- El ataque potencial: Si una aplicación valida firmas de manera laxa o permite discrepancias en los parámetros criptográficos (como aceptar hashes más débiles o no validar la longitud exacta del mensaje), un atacante podría interceptar una transferencia legítima (ej. "Transfer 100 euros") y modificarla en tránsito (ej. añadiendo ceros o cambiando destinatarios) logrando que una verificación mal diseñada dé el visto bueno (falsificación o modificación de datos).

c) Modifica el código para que no haya el problema identificado en b) e indica qué ocurre si el mensaje cambia, aunque sea un byte

In [3]:
# VERIFICACIÓN CORREGIDA
try:
    public_key.verify(
        signature,
        message,               # Usamos el mensaje original exacto
        padding.PKCS1v15(),
        hashes.SHA256()        # Usamos el mismo algoritmo hash (SHA256)
    )
    print("Firma válida: El mensaje es auténtico y no ha sido alterado.")
except Exception as e:
    print("Firma inválida:", e)

Firma válida: El mensaje es auténtico y no ha sido alterado.


¿Qué ocurre si el mensaje cambia aunque sea un byte?

- Debido a la propiedad matemática de las funciones hash (conocida como efecto avalancha), si modificas un solo byte del mensaje (por ejemplo, cambiar 100 por 200), el hash SHA-256 resultante cambiará por completo. Al intentar verificarlo, el hash recalculado no coincidirá con el hash obtenido al descifrar la firma con la clave pública, por lo que saltará una excepción y el programa imprimirá "Firma inválida".

d) ¿Es PKCS1v15 una buena alternativa? De no ser así ¿Qué se utiliza?

- No es la mejor alternativa para implementaciones modernas. El esquema de relleno PKCS#1 v1.5 es vulnerable a ciertos ataques avanzados (como ataques de relleno o Padding Oracle Attacks) si no se implementa con contramedidas estrictas.

- Alternativa actual: Hoy en día se recomienda encarecidamente utilizar PSS (Probabilistic Signature Scheme) (padding.PSS), que incorpora aleatoriedad (un salt) en el proceso de relleno, ofreciendo una seguridad demostrable mucho más sólida frente a ataques criptográficos.

e) ¿Podría un atacante modificar el mensaje y aun así pasar la verificación en algún escenario mal diseñado?

- Sí, podría ocurrir en escenarios con fallos graves de diseño:

    - Manejo incorrecto de excepciones: Si el código del servidor captura la excepción de fallo de firma de manera genérica y aun así permite continuar con la ejecución del programa.

    - Colisiones de Hash: Si se utilizasen funciones hash obsoletas y vulnerables a colisiones (como MD5 o SHA-1), un atacante podría calcular dos mensajes distintos que produzcan el mismo hash. (Afortunadamente, SHA-256 es seguro frente a esto hoy en día).

    - Reutilización de parámetros o claves débiles.